# Pengujian Sistem Machine Learning di Cloud

## Prediction Request ke Model Serving — Proyek Akhir MLOps

**Username Dicoding:** sonnyariady

Notebook ini digunakan untuk menguji dan melakukan **prediction request** ke sistem machine learning yang telah dijalankan di cloud (saran penilaian ke-3).

In [1]:
import csv
import os
import random

import requests

# Ganti dengan URL web app yang sudah di-deploy (Railway/Heroku).
APP_URL = os.environ.get("APP_URL", "https://abalone-mlops-dicoding-production.up.railway.app")
PREDICT_URL = APP_URL.rstrip("/") + "/predict"

random.seed(42)

In [2]:
DATA_FILE = os.path.join("data", "abalone.csv")

with open(DATA_FILE, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))

print("Total baris dataset :", len(rows))
sample = random.sample(rows, 5)
print("Mengambil 5 contoh acak untuk pengujian...")

Total baris dataset : 4177
Mengambil 5 contoh acak untuk pengujian...


## Contoh Prediction Request

Mengirim 5 contoh acak dari dataset ke endpoint `/predict` dalam satu request (batch).

In [3]:
payload = {
    "features": [{k: v for k, v in row.items() if k != "label"} for row in sample]
}

response = requests.post(PREDICT_URL, json=payload, timeout=60)
print("Status code :", response.status_code)
print()
print(response.json())

Status code : 200

{'predictions': [{'label': 'dewasa', 'prediction': 1, 'probability': 0.5346}, {'label': 'dewasa', 'prediction': 1, 'probability': 0.533}, {'label': 'dewasa', 'prediction': 1, 'probability': 0.5323}, {'label': 'dewasa', 'prediction': 1, 'probability': 0.5492}, {'label': 'dewasa', 'prediction': 1, 'probability': 0.5059}]}


## Uji Akurasi Model di Cloud

Melakukan prediksi pada 100 sampel dan membandingkannya dengan label aktual untuk mengukur akurasi model yang berjalan di cloud.

In [4]:
test_sample = random.sample(rows, 100)
payload = {
    "features": [{k: v for k, v in row.items() if k != "label"} for row in test_sample]
}

response = requests.post(PREDICT_URL, json=payload, timeout=120)
assert response.status_code == 200, response.text

predictions = response.json()["predictions"]
actual = [int(row["label"]) for row in test_sample]
predicted = [item["prediction"] for item in predictions]

correct = sum(1 for a, p in zip(actual, predicted) if a == p)
accuracy = correct / len(actual)

print(f"Akurasi model di cloud : {accuracy:.2%} ({correct}/{len(actual)})")

Akurasi model di cloud : 50.00% (50/100)


## Cara Menjalankan

```bash
# Atur URL web app deployment Anda
APP_URL=https://<url-web-app>.railway.app jupyter notebook sonnyariady-testing.ipynb
```

Apabila dijalankan tanpa variabel `APP_URL`, notebook menggunakan URL default pada sel pertama (ganti sesuai deployment Anda).